# Legislative Content Extraction (ADK) - Evaluation Smoke Test

Fetch the dataset from Langfuse and run the agent on the first item to verify the pipeline works end-to-end.

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv

# Ensure cwd is the repo root so relative paths work
REPO_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd()
# Walk up until we find pyproject.toml
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

load_dotenv(verbose=True)
print(f"Working directory: {Path.cwd()}")


Working directory: /Users/a6137962/eval-agents
SSL: macOS system keychain injected via truststore


## 1. Fetch dataset from Langfuse

In [8]:
from aieng.agent_evals.async_client_manager import AsyncClientManager

client_manager = AsyncClientManager.get_instance()
langfuse_client = client_manager.langfuse_client

DATASET_NAME = "legislative-docs-v2"
FILES_DIR = Path("implementations/legislative_content_extraction/files").resolve()
MAX_CONCURRENCY = 3

dataset = langfuse_client.get_dataset(DATASET_NAME)
print(f"Dataset name: {dataset.name}")
print(f"Items:   {len(dataset.items)}")
print(f"Files dir:     {FILES_DIR}")
print(f"Concurrency:   {MAX_CONCURRENCY}")

Dataset name: legislative-docs-v2
Items:   26
Files dir:     /Users/a6137962/eval-agents/implementations/legislative_content_extraction/files
Concurrency:   3


## 2. Inspect the first item

In [9]:
first_item = dataset.items[0]

print("Input:")
print(json.dumps(first_item.input, indent=2))
print("\nExpected output:")
print(json.dumps(first_item.expected_output, indent=2))
print("\nMetadata:")
print(json.dumps(first_item.metadata, indent=2))

Input:
{
  "prompt": "Extract legislative metadata from this bill.",
  "record_id": "MI_SB0002",
  "pdf_file_name": "2025-SIB-0002.pdf",
  "html_page_link": "https://www.legislature.mi.gov/Bills/Bill?ObjectName=2025-SB-0002"
}

Expected output:
{
  "title": "A bill to amend 1976 PA 442, entitled \"Freedom of information act,\" by amending sections 6, 10, and 13 (MCL 15.236, 15.240, and 15.243), section 6 as amended by 1996 PA 553, section 10 as amended by 2014 PA 563, and section 13 as amended by 2023 PA 64, and by adding section 14a.",
  "summary": "Amends Michigan's Freedom of Information Act to establish FOIA coordinator designations for the house and senate, create legislative appeal procedures for denials, revise judicial review and remedies, expand exemptions applicable to executive-office and legislative records, preserve access to legislative salary records, and add section 14a stating that FOIA's application to state legislative public bodies does not limit constitutional legi

## 3. Run evaluation on the dataset

In [10]:
from aieng.agent_evals.legislative_content_extraction.evaluation.offline import evaluate
from datetime import datetime, timezone
iso_timestamp = datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

# add name to identify your run
custom_run_name = "eval"

# add description to identify your run
custom_description = "round-eval-day-3"

await evaluate(
    dataset_name=DATASET_NAME,
    files_dir=FILES_DIR,
    max_concurrency=MAX_CONCURRENCY,
    run_name=f"{custom_run_name} - {iso_timestamp}",
    description=custom_description
)

2026-04-06 15:50:36,537 INFO aieng.agent_evals.legislative_content_extraction.agent: Extracting metadata from: /Users/a6137962/eval-agents/implementations/legislative_content_extraction/files/MI_SB0002/2025-SIB-0002.pdf
2026-04-06 15:50:36,566 INFO google_adk.google.adk.models.google_llm: Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
2026-04-06 15:50:36,704 INFO aieng.agent_evals.legislative_content_extraction.agent: Extracting metadata from: /Users/a6137962/eval-agents/implementations/legislative_content_extraction/files/MI_SB0001/2025-SIB-0001.pdf
2026-04-06 15:50:36,726 INFO google_adk.google.adk.models.google_llm: Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
2026-04-06 15:50:36,728 INFO aieng.agent_evals.legislative_content_extraction.agent: Extracting metadata from: /Users/a6137962/eval-agents/implementations/legislative_content_extraction/files/MI_HB4007/2025-HIB-4007.pdf
2026

Evaluating traces ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26/26 0:00:00 0:03:41m 0:03:41


2026-04-06 15:57:51,292 INFO aieng.agent_evals.evaluation.trace: Trace 987f47376ff0a240c728e27dc773540b evaluation skipped: Trace did not become ready within wait window.
2026-04-06 15:57:51,294 INFO aieng.agent_evals.legislative_content_extraction.evaluation.offline: Evaluation complete. 26 items processed.
